In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os

# Add project root to sys.path so that src/ can be imported
PROJECT_ROOT = '/content/drive/MyDrive/traffic-learning-system'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import ListedColormap
from IPython.display import HTML, display
import glob
import json as _json

In [ ]:
file_paths = glob.glob('/content/drive/MyDrive/Docs/*.xlsx')
print(file_paths)
num_blocks = len(file_paths)

## Part 1: Environment
Imports the grid road-network environment from `src/environment.py`.

In [ ]:
from src.environment import get_state_index, Table

## Part 2: LSTM Prediction Service
Loads `get_directional_predictions()` from `lstm_traffic_prediction.ipynb`
without triggering the full LSTM training run.

In [ ]:
def _load_function_from_notebook(nb_path, func_name):
    with open(nb_path, 'r', encoding='utf-8') as f:
        nb = _json.load(f)
    for cell in nb['cells']:
        if cell['cell_type'] != 'code':
            continue
        src = ''.join(cell['source'])
        if f'def {func_name}(' in src:
            ns = {}
            exec(compile(src, nb_path, 'exec'), ns)
            return ns[func_name]
    raise ValueError(f"Function '{func_name}' not found in {nb_path}")

PREDICT_NB_PATH = (
    '/content/drive/MyDrive/traffic-learning-system/notebooks/'
    'lstm_traffic_prediction.ipynb'
)
get_directional_predictions = _load_function_from_notebook(
    PREDICT_NB_PATH, 'get_directional_predictions')
print(f'Loaded get_directional_predictions from {PREDICT_NB_PATH}')

In [ ]:
DATA_DIR         = '/content/drive/MyDrive/Docs/'
PREDICTION_START = '2024-04-03 18:00:00'
LOOK_BACK        = 12

print(f'Requesting LSTM predictions (data_dir={DATA_DIR!r}) ...')
prediction_result = get_directional_predictions(
    data_dir=DATA_DIR,
    prediction_start_time=PREDICTION_START,
    look_back=LOOK_BACK,
)
lstm_by_intersection = prediction_result['by_intersection']
print(f'\nLSTM predictions ready for {len(lstm_by_intersection)} intersections:')
for name, arr in lstm_by_intersection.items():
    print(f'  {name}: shape {arr.shape}')

## Part 3: Data Processing
Processes each intersection's xlsx data and builds numpy arrays
consumed by the Q-learning agent.

In [ ]:
def process_intersection(group_name, group_df, full_index,
                         lstm_by_intersection=None):
    flow_5min = (
        group_df
        .groupby('方向')['车牌号']
        .resample('5min')
        .count()
        .unstack(level=0)
    )
    flow_5min = flow_5min.reindex(full_index).fillna(0)
    for d in [1, 2, 3, 4]:
        if d not in flow_5min.columns:
            flow_5min[d] = 0
    flow_5min   = flow_5min[[1, 2, 3, 4]]
    data_matrix = flow_5min.values
    means = np.mean(data_matrix, axis=0)
    vars_ = np.var(data_matrix,  axis=0)
    if lstm_by_intersection is not None and group_name in lstm_by_intersection:
        lstm_pred = lstm_by_intersection[group_name]
    else:
        lstm_pred = np.zeros((12, 4))
        print(f'[RL] Warning: no LSTM prediction for {group_name!r}. Using zeros.')
    return {'name': group_name, 'lstm': lstm_pred,
            'mean': means, 'var': vars_, 'last_step': data_matrix[-1]}


full_index = pd.date_range(start='2024-04-01T00:00:00',
                           end='2024-04-06T19:00:00', freq='5min')
results = []
for path in file_paths:
    name = os.path.basename(path).replace('.xlsx', '')
    df   = pd.read_excel(path)
    df['时间'] = pd.to_datetime(df['时间'])
    df.set_index('时间', inplace=True)
    results.append(process_intersection(name, df, full_index, lstm_by_intersection))

lstm_lst  = np.array([r['lstm'].T    for r in results])
mean_lst  = np.array([r['mean']      for r in results])
var_lst   = np.array([r['var']       for r in results])
queue_lst = np.array([r['last_step'] for r in results])
print(f'Processed {len(results)} intersections.')
print(f'  lstm_lst  : {lstm_lst.shape}')
print(f'  mean_lst  : {mean_lst.shape}')
print(f'  var_lst   : {var_lst.shape}')
print(f'  queue_lst : {queue_lst.shape}')

## Part 4: Q-Learning & Visualisation
Imports training loop and visualisation utilities from `src/`.

In [ ]:
from src.rl_agent     import q_learning_train, plot_paths
from src.visualization import (
    plot_rewards,
    compute_greedy_trajectory,
    animate_learned_policy_pretty,
)

## Part 5: Training Run

In [ ]:
ns_num = 3
ew_num = num_blocks // ns_num
street = np.zeros((ns_num, ew_num))

# Grid layout:
#  0 |  1 |  2 |  3
# ----+----+----+----
#  4 |  5 |  6 |  7
# ----+----+----+----
#  8 |  9 | 10 | 11

Episodes      = 500
Max_steps     = 100
Dist_Weight   = 0.1
Cong_Weight   = 0.5
Dec_Weight    = 0.95
Learning_Rate = 0.1
Start_loc     = 4
End_loc       = 11
Epsilon_start = 1.0
Epsilon_min   = 0.05
Epsilon_decay = 0.99

policy_lst = np.ones([12, 4])
env = Table(street, policy_lst)

q_table, rewards, path_tot, path_dir = q_learning_train(
    env=env, queue_maps=queue_lst, mean_maps=mean_lst,
    var_maps=var_lst, lstm_maps=lstm_lst,
    epochs=Episodes, max_steps_per_episode=Max_steps,
    alpha=Dist_Weight, beta=Cong_Weight, gamma=Dec_Weight,
    lamda=Learning_Rate, start_loc=Start_loc, end_loc=End_loc,
    epsilon_start=Epsilon_start, epsilon_min=Epsilon_min,
    epsilon_decay=Epsilon_decay,
)

plot_rewards(rewards, window=20)

animate_learned_policy_pretty(
    env, Start_loc, q_table,
    end_pos=End_loc, max_steps=Max_steps, interval=400,
)

## Part 6: Learning Rate Sensitivity Analysis
200 independent trials per learning rate — Top-10 path distribution.

In [ ]:
import collections

N_RUNS         = 200
LEARNING_RATES = [0.1, 0.01, 0.001]
TOP_N          = 10

def _fmt_path(p):
    return 'NULL' if p == 'NULL' else ' -> '.join(str(n) for n in p)

path_counters = {}
for lr in LEARNING_RATES:
    counter = collections.Counter()
    print(f'\n[lr={lr}] Running {N_RUNS} trials ...', flush=True)
    for _ in range(N_RUNS):
        env_run = Table(street, policy_lst)
        _, _, pt, _ = q_learning_train(
            env=env_run, queue_maps=queue_lst, mean_maps=mean_lst,
            var_maps=var_lst, lstm_maps=lstm_lst,
            epochs=Episodes, max_steps_per_episode=Max_steps,
            alpha=Dist_Weight, beta=Cong_Weight, gamma=Dec_Weight,
            lamda=lr, start_loc=Start_loc, end_loc=End_loc,
            epsilon_start=Epsilon_start, epsilon_min=Epsilon_min,
            epsilon_decay=Epsilon_decay,
        )
        reached = (len(pt) > 1 and int(pt[-1]) == End_loc)
        key = tuple(int(n) for n in pt) if reached else 'NULL'
        counter[key] += 1
    path_counters[lr] = counter
    print(f'  Done. Unique outcomes: {len(counter)}')

print('\n' + '='*60 + '\nSUMMARY\n' + '='*60)
for lr in LEARNING_RATES:
    print(f'\nLearning Rate = {lr}:')
    for k, cnt in path_counters[lr].most_common():
        print(f'  {_fmt_path(k):<55s}  x{cnt}')

colors = {'0.1': '#2196f3', '0.01': '#4caf50', '0.001': '#ff9800'}
for lr in LEARNING_RATES:
    counter    = path_counters[lr]
    top        = counter.most_common(TOP_N)
    keys       = [k for k, _ in top]
    counts     = [v for _, v in top]
    labels     = [_fmt_path(k) for k in keys]
    total_runs = sum(counter.values())
    others     = total_runs - sum(counts)
    fig, ax = plt.subplots(figsize=(max(6, len(keys)*1.4), 6))
    bars = ax.bar(range(len(keys)), counts,
                  color=colors[str(lr)], edgecolor='black',
                  linewidth=0.8, alpha=0.88)
    for bar, cnt in zip(bars, counts):
        pct = cnt / total_runs * 100
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
                f'{cnt}\n({pct:.1f}%)', ha='center', va='bottom',
                fontsize=9, fontweight='bold')
    ax.set_xticks(range(len(keys)))
    ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
    ax.set_xlabel('Path', fontsize=12)
    ax.set_ylabel('Count  (out of 200 runs)', fontsize=12)
    ax.set_title(
        f'Top {TOP_N} Paths - Learning Rate = {lr}\n'
        f'(Start {Start_loc} -> End {End_loc}, {Episodes} eps/run, {N_RUNS} trials'
        + (f', {others} runs in other paths' if others > 0 else '') + ')',
        fontsize=13, fontweight='bold')
    ax.set_ylim(0, N_RUNS+20)
    ax.grid(axis='y', linestyle='--', alpha=0.45)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()